## Missing Values
### Links:
[Tutorial](https://www.kaggle.com/code/alexisbcook/missing-values/tutorial)
[Exercise](https://www.kaggle.com/code/marcopolo1313/exercise-missing-values/edit)

The Kaggle section already has the basic syntaxes which I am supposed to paste here and expand more on it since I prefer running on my local device to kaggle's servers


#### Simple Setup

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
# Read the data
X_full = pd.read_csv(r'C:\Users\Anwesh\Desktop\Py\MachineLearning\KaggleML-Intermediate\DatasetsKaggleML-Intermediate\train.csv', index_col='Id')
X_test_full = pd.read_csv(r'C:\Users\Anwesh\Desktop\Py\MachineLearning\KaggleML-Intermediate\DatasetsKaggleML-Intermediate\test.csv', index_col='Id')

# Remove rows with missing target, separate target from predictors
X_full.dropna(axis=0, subset=['SalePrice'], inplace=True) #axis=0 means it tells pandas to drop the entire empty rows
y = X_full.SalePrice #Save the Predictor in a series called y
X_full.drop(['SalePrice'], axis=1, inplace=True) #delete the predictor from X_full df

# To keep things simple, we'll use only numerical predictors
X = X_full.select_dtypes(exclude=['object'])
X_test = X_test_full.select_dtypes(exclude=['object'])

# Break off validation set from training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2,
                                                      random_state=0)
print(X_train.head())
print(X_train.shape)
no_of_missing_vals_inEachColumn = X_train.isnull().sum()
print(no_of_missing_vals_inEachColumn[no_of_missing_vals_inEachColumn>0])


     MSSubClass  LotFrontage  LotArea  OverallQual  OverallCond  YearBuilt  \
Id                                                                           
619          20         90.0    11694            9            5       2007   
871          20         60.0     6600            5            5       1962   
93           30         80.0    13360            5            7       1921   
818          20          NaN    13265            8            5       2002   
303          20        118.0    13704            7            5       2001   

     YearRemodAdd  MasVnrArea  BsmtFinSF1  BsmtFinSF2  ...  GarageArea  \
Id                                                     ...               
619          2007       452.0          48           0  ...         774   
871          1962         0.0           0           0  ...         308   
93           2006         0.0         713           0  ...         432   
818          2002       148.0        1218           0  ...         857   
303      

C:\Users\Anwesh\AppData\Local\Temp\ipykernel_12008\1572560414.py:8: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  X_full.dropna(axis=0, subset=['SalePrice'], inplace=True) #axis=0 means it tells pandas to drop the entire empty rows
C:\Users\Anwesh\AppData\Local\Temp\ipykernel_12008\1572560414.py:10: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  X_full.drop(['SalePrice'], axis=1, inplace=True) #delete the predictor from X_full df
C:\Users\Anwesh\AppData\Local\Temp\ipykernel_12008\1572560414.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are excluded by select_dtypes when 'object' dtype is specified. This behavior is deprecated and

In [13]:
# Fill in the line below: How many rows are in the training data?
num_rows = 1168

# Fill in the line below: How many columns in the training data
# have missing values?
num_cols_with_missing = 3

# Fill in the line below: How many missing entries are contained in 
# all of the training data?
tot_missing = (212+6+58)

# Check your answers
#step_1.a.check()-->>Was Correct

#### Define score_dataset

In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Function for comparing different approaches
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=44)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

#### Drop Empty Columns

In [21]:
# Fill in the line below: get names of columns with missing values
 # Your code here

# Fill in the lines below: drop columns in training and validation data
list_of_missing_columns = [columns_ for columns_ in X_train.columns if X_train[columns_].isnull().any()] 
# .isnull() will return a series of booleans but .any() will return a cumulative true or false value for the if statement to evaluate
reduced_X_train = X_train.drop(list_of_missing_columns, axis=1) #axis 1 means columns btw if you don't know yet
reduced_X_valid = X_valid.drop(list_of_missing_columns,axis=1)

# Check your answers
#step_2.check()--> Correct

print("Mean Absolute error when dropping NaN columns: ", end="")
print(score_dataset(reduced_X_train, reduced_X_valid, y_train, y_valid))


Mean Absolute error when dropping NaN columns: 17906.554828767123


#### Imputation 
[Geek for Geeks SimpleImputer Tutorial](https://www.geeksforgeeks.org/machine-learning/ml-handle-missing-data-with-simple-imputer/)

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean') #Default strategy is mean and default missing_values is np.NaN

# I could just use fit_transform() instead of fit() and transform() separately
imputer.fit(X_train)
imputed_X_train = pd.DataFrame(imputer.transform(X_train))
imputed_X_valid = pd.DataFrame(imputer.transform(X_valid))

imputed_X_train.columns = X_train.columns
imputed_X_valid.columns = X_valid.columns
print(type(imputed_X_train))
# previous mistake->> Incorrect: imputed_X_train is not a pandas DataFrame.
# previous mistake ->> I had fitted test data too, but fitting training data is enough for both X_train and X_valid
#I need to have these values as df and not 

#step_3.a.check() --> Correct! Finally

<class 'pandas.DataFrame'>


In [6]:
print("Mean Absolute Error from Imputation Method:",end="")
print(score_dataset(imputed_X_train,imputed_X_valid,y_train,y_valid))

Mean Absolute Error from Imputation Method:18149.141632420095


#### Generate Text Predictions

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')
final_X_train = pd.DataFrame(imputer.fit_transform(X_train))
final_X_valid = pd.DataFrame(imputer.transform(X_valid))
#step_4.a.check()-->Correct

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
myModel = RandomForestRegressor(n_estimators=150, random_state=44)
myModel.fit(final_X_train, y_train) #training data
preds_valid = myModel.predict(final_X_valid) #predict from our final imputed final_X_valid training dataset
print("Mean Absolute Error:", end="")
print(mean_absolute_error(y_valid, preds_valid))

Mean Absolute Error:18232.22180365297


In [ ]:
final_X_test = pd.DataFrame(imputer.transform(X_test))
preds_test = myModel.predict(final_X_test)
#step_4.b.check() -->> Correct

For Submission to Kaggle:

In [ ]:
# Save test predictions to file
output = pd.DataFrame({'Id': X_test.index,
                       'SalePrice': preds_test})
output.to_csv('submission.csv', index=False)